# Custom Formatter
> Build your own formatter using the `Formatter` protocol.

Anchor's `Formatter` is a Python `Protocol`: any class with
`format_type` and `format()` is a valid formatter — no
inheritance required.

**Time:** ~10 minutes

## Setup

In [ ]:
from anchor.formatters import Formatter
from anchor.models import ContextWindow, ContextItem, SourceType

## Build a Sample Context Window

In [ ]:
from anchor.models import ContextWindow, ContextItem, SourceType

# Build a sample context window with mixed sources
window = ContextWindow(max_tokens=4096)
window.add_item(ContextItem(
    id="sys-1",
    content="You are helpful.",
    source=SourceType.SYSTEM,
    score=1.0,
    priority=10,
    token_count=5,
))
window.add_item(ContextItem(
    id="doc-1",
    content="Python is a language.",
    source=SourceType.RETRIEVAL,
    score=0.9,
    priority=5,
    token_count=6,
))

print(f"Context window: {len(window.items)} items, "
      f"budget {window.max_tokens} tokens")
for item in window.items:
    print(f"  [{item.source.value}] {item.id}: {item.content!r}")

## Define an XML Formatter
Implement the two required members:
- `format_type` (read-only property) — a string identifier.
- `format(window)` — accepts a `ContextWindow`, returns any type.

In [ ]:
class XMLFormatter:
    """Custom formatter that outputs XML."""

    @property
    def format_type(self) -> str:
        return "xml"

    def format(self, window: ContextWindow) -> str:
        lines = ["<context>"]
        for item in window.items:
            lines.append(
                f'  <item source="{item.source.value}" '
                f'priority="{item.priority}">'
            )
            lines.append(f"    {item.content}")
            lines.append("  </item>")
        lines.append("</context>")
        return "\n".join(lines)

print("XMLFormatter defined.")

## Verify Protocol Compliance
`isinstance()` checks work because `Formatter` is a
`runtime_checkable` Protocol.

In [ ]:
xml_fmt = XMLFormatter()
print(f"Is a Formatter? {isinstance(xml_fmt, Formatter)}")
print(f"Format type:   {xml_fmt.format_type}")

## Generate XML Output

In [ ]:
xml_output = xml_fmt.format(window)
print(xml_output)

## Another Example: CSV Formatter
The protocol is minimal, so you can target any serialisation format.

In [ ]:
class CSVFormatter:
    """Formatter that outputs comma-separated values."""

    @property
    def format_type(self) -> str:
        return "csv"

    def format(self, window: ContextWindow) -> str:
        rows = ["id,source,priority,content"]
        for item in window.items:
            rows.append(
                f"{item.id},{item.source.value},"
                f"{item.priority},{item.content}"
            )
        return "\n".join(rows)

csv_fmt = CSVFormatter()
print(f"Is a Formatter? {isinstance(csv_fmt, Formatter)}")
print()
print(csv_fmt.format(window))

## Swap Formatters at Runtime
Because every formatter shares the same protocol, you can select
one dynamically.

In [ ]:
from anchor.formatters import AnthropicFormatter, GenericTextFormatter

formatters = {
    "anthropic": AnthropicFormatter(),
    "text": GenericTextFormatter(),
    "xml": XMLFormatter(),
    "csv": CSVFormatter(),
}

for name, fmt in formatters.items():
    result = fmt.format(window)
    preview = str(result)[:60].replace("\n", " ")
    print(f"{name:10s} -> {type(result).__name__:5s} | {preview}...")

## Key Takeaways

- `Formatter` is a `runtime_checkable` Protocol — no base class needed.
- Implement `format_type` (property) and `format(window)` to comply.
- Custom formatters plug into the same pipeline as built-in ones.
- Use `isinstance(obj, Formatter)` to verify compliance at runtime.